# P2.3 — Simulasi Perambatan Banjir: Metode Muskingum
### Studio Pemrograman Python | Teknik Sipil

---

**Skenario:**  
Stasiun Pengukuran Debit (SPD) di **hulu Sungai Bengawan Solo** mencatat gelombang banjir
dengan debit puncak 500 m³/s. Anda diminta memperkirakan **kapan dan seberapa besar**
banjir itu tiba di stasiun **hilir** sejauh 50 km, sehingga dapat dikeluarkan peringatan
dini kepada masyarakat.

Gunakan **Metode Muskingum** untuk memodelkan perambatan gelombang banjir.

---

## Teori: Metode Muskingum

Penyimpanan air di segmen sungai:
$$S = K\,[X\,I + (1-X)\,O]$$

Persamaan routing eksplisit (maju waktu):
$$O_{t+1} = C_0\,I_{t+1} + C_1\,I_t + C_2\,O_t$$

Koefisien:
$$C_0 = \frac{-KX + 0.5\Delta t}{K(1-X)+0.5\Delta t}, \quad
C_1 = \frac{KX + 0.5\Delta t}{K(1-X)+0.5\Delta t}, \quad
C_2 = \frac{K(1-X)-0.5\Delta t}{K(1-X)+0.5\Delta t}$$

**Syarat:** $C_0 + C_1 + C_2 = 1$ (cek ini untuk validasi!)

| Simbol | Keterangan | Satuan |
|--------|-----------|--------|
| $K$    | Parameter penyimpanan sungai | jam |
| $X$    | Faktor pembobot ($0 \leq X \leq 0.5$) | — |
| $\Delta t$ | Interval waktu iterasi | jam |
| $I$    | Debit masuk (inflow) hulu | m³/s |
| $O$    | Debit keluar (outflow) hilir | m³/s |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
print('Library siap!')

---
## Step 1: Parameter Model & Data Inflow

In [ ]:
# ── Parameter Muskingum ───────────────────────────────────────────────────────
K  = 5.0   # jam  — parameter penyimpanan sungai
X  = 0.2   # —    — faktor pembobot (0 = reservoir, 0.5 = kinematic wave)
dt = 1.0   # jam  — interval waktu iterasi

# ── Data Inflow dari Stasiun Hulu (m³/s) ─────────────────────────────────────
I = np.array([50, 100, 200, 300, 400, 500, 400, 300, 200, 100, 50], dtype=float)
# Jam 0  sampai jam 10

O0 = 50.0   # debit awal di hilir (m³/s) — asumsi = inflow awal

waktu = np.arange(len(I))   # jam 0 s.d. 10

print(f'Jumlah titik waktu : {len(I)}')
print(f'Debit inflow puncak: {I.max():.0f} m³/s pada jam ke-{I.argmax()}')

---
## Step 2: Hitung Koefisien Muskingum C0, C1, C2

In [ ]:
# Penyebut bersama
D = K * (1 - X) + 0.5 * dt

# TODO: Hitung C0, C1, C2 menggunakan rumus di atas
C0 = (-K * X + 0.5 * dt) / D
C1 = ( K * X + 0.5 * dt) / D
C2 = ( K * (1 - X) - 0.5 * dt) / D

print(f'C0 = {C0:.4f}')
print(f'C1 = {C1:.4f}')
print(f'C2 = {C2:.4f}')
print(f'C0 + C1 + C2 = {C0+C1+C2:.4f}  (harus = 1.0000)')

# Validasi
assert abs(C0 + C1 + C2 - 1.0) < 1e-9, 'ERROR: jumlah koefisien tidak sama dengan 1!'
assert C0 >= 0, 'WARNING: C0 negatif — cek nilai K, X, dan dt!'
print('\nValidasi LULUS — koefisien sudah benar!')

---
## Step 3: Routing — Hitung Outflow O(t)

Iterasi maju waktu: gunakan $O_t$ untuk menghitung $O_{t+1}$.

In [ ]:
n  = len(I)
O  = np.zeros(n)
O[0] = O0   # kondisi awal

# TODO: Lengkapi loop routing berikut
# Persamaan: O[t] = C0*I[t] + C1*I[t-1] + C2*O[t-1]
for t in range(1, n):
    O[t] = C0 * I[t] + C1 * I[t - 1] + C2 * O[t - 1]   # <-- ini benar, pahami logikanya!

# Analisis reduksi puncak
Q_peak_in  = I.max()
Q_peak_out = O.max()
t_peak_in  = I.argmax()
t_peak_out = O.argmax()
lag        = t_peak_out - t_peak_in
reduksi    = (1 - Q_peak_out / Q_peak_in) * 100

print(f'Inflow puncak   : {Q_peak_in:.1f} m³/s  (jam ke-{t_peak_in})')
print(f'Outflow puncak  : {Q_peak_out:.1f} m³/s  (jam ke-{t_peak_out})')
print(f'Keterlambatan   : {lag} jam')
print(f'Reduksi puncak  : {reduksi:.1f}%')

---
## Step 4: Tabel Hasil

In [ ]:
hasil = pd.DataFrame({
    'Jam'                    : waktu,
    'Inflow I (m³/s)'        : I,
    'Outflow O (m³/s)'       : O.round(2),
    'Redaman ΔQ (m³/s)'      : (I - O).round(2)
})
hasil.set_index('Jam', inplace=True)
print(hasil.to_string())

---
## Step 5: Visualisasi Perambatan Gelombang Banjir

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(waktu, I, alpha=0.12, color='steelblue')
ax.fill_between(waktu, O, alpha=0.12, color='tomato')
ax.plot(waktu, I, 'steelblue', linewidth=2.5, marker='o', markersize=6,
        label=f'Inflow Hulu  (Qpeak = {Q_peak_in:.0f} m³/s, jam ke-{t_peak_in})')
ax.plot(waktu, O, 'tomato',    linewidth=2.5, marker='s', markersize=6,
        label=f'Outflow Hilir (Qpeak = {Q_peak_out:.1f} m³/s, jam ke-{t_peak_out})')

# Anotasi keterlambatan
ax.annotate('', xy=(t_peak_out, Q_peak_out), xytext=(t_peak_in, Q_peak_in),
            arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))
ax.text((t_peak_in + t_peak_out) / 2, max(Q_peak_in, Q_peak_out) * 0.97,
        f'Lag = {lag} jam', ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('Waktu (jam)', fontsize=12)
ax.set_ylabel('Debit (m³/s)', fontsize=12)
ax.set_title(f'Perambatan Gelombang Banjir — Metode Muskingum\n'
             f'K={K} jam, X={X}, Δt={dt} jam',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, len(waktu) - 1)

plt.tight_layout()
plt.savefig('P2.3_muskingum.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafik disimpan sebagai P2.3_muskingum.png')

---
## Level 2 — Analisis Sensitivitas Parameter K dan X

Ubah nilai K dan X untuk melihat bagaimana sungai dengan karakteristik berbeda
memperambatkan gelombang banjir.

In [ ]:
# Variasi parameter K (efek penyimpanan)
K_list  = [2, 5, 8]      # jam — K kecil = sungai pendek/cepat, K besar = lambat
X_fixed = 0.2
colors  = ['seagreen', 'steelblue', 'tomato']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for K_var, warna in zip(K_list, colors):
    D_var  = K_var * (1 - X_fixed) + 0.5 * dt
    C0_v = (-K_var * X_fixed + 0.5 * dt) / D_var
    C1_v = ( K_var * X_fixed + 0.5 * dt) / D_var
    C2_v = ( K_var * (1 - X_fixed) - 0.5 * dt) / D_var

    O_v = np.zeros(n)
    O_v[0] = O0
    for t in range(1, n):
        O_v[t] = C0_v * I[t] + C1_v * I[t-1] + C2_v * O_v[t-1]

    ax1.plot(waktu, O_v, color=warna, linewidth=2,
             label=f'K={K_var} jam  (Qpeak={O_v.max():.1f} m³/s)')

ax1.plot(waktu, I, 'k--', linewidth=1.5, label='Inflow')
ax1.set_title(f'Variasi K  (X = {X_fixed})', fontsize=12, fontweight='bold')
ax1.set_xlabel('Waktu (jam)')
ax1.set_ylabel('Debit Outflow (m³/s)')
ax1.legend(fontsize=9)

# Variasi parameter X (efek pembobot)
X_list  = [0.0, 0.2, 0.4]   # X=0 reservoir murni, X=0.5 kinematic wave
K_fixed = 5.0

for X_var, warna in zip(X_list, colors):
    D_var  = K_fixed * (1 - X_var) + 0.5 * dt
    C0_v = (-K_fixed * X_var + 0.5 * dt) / D_var
    C1_v = ( K_fixed * X_var + 0.5 * dt) / D_var
    C2_v = ( K_fixed * (1 - X_var) - 0.5 * dt) / D_var

    O_v = np.zeros(n)
    O_v[0] = O0
    for t in range(1, n):
        O_v[t] = C0_v * I[t] + C1_v * I[t-1] + C2_v * O_v[t-1]

    ax2.plot(waktu, O_v, color=warna, linewidth=2,
             label=f'X={X_var}  (Qpeak={O_v.max():.1f} m³/s)')

ax2.plot(waktu, I, 'k--', linewidth=1.5, label='Inflow')
ax2.set_title(f'Variasi X  (K = {K_fixed} jam)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Waktu (jam)')
ax2.set_ylabel('Debit Outflow (m³/s)')
ax2.legend(fontsize=9)

plt.suptitle('Analisis Sensitivitas Parameter Muskingum', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# TODO — Pertanyaan Diskusi:
# 1. Parameter K atau X yang lebih berpengaruh terhadap reduksi puncak banjir?
# 2. Sungai dengan K besar dan X kecil cocok untuk tipe sungai seperti apa?
# 3. Bagaimana cara mengestimasi K dari data pengukuran debit riil?

---
## Level 3 — Baca Data dari CSV

Sel di bawah menunjukkan cara mengganti data sintetik dengan data riil dari file CSV.

In [ ]:
# TODO: Uncomment dan sesuaikan path file CSV Anda

# import pandas as pd
# df_csv = pd.read_csv('data/inflow_hulu.csv')   # kolom: 'Jam', 'Inflow_m3s'
# I = df_csv['Inflow_m3s'].values.astype(float)
# waktu = df_csv['Jam'].values
# O0 = I[0]   # asumsi kondisi awal = inflow pertama
#
# print(f'Data dimuat: {len(I)} titik waktu, inflow puncak = {I.max():.1f} m³/s')

# Setelah memuat data, jalankan kembali sel Step 2 dan Step 3 di atas.
print('Sel ini siap digunakan — hapus tanda # untuk mengaktifkan.')